# KOveri Primer Designer

Interactive workflow for designing CRISPR KO genotyping primers around user-provided gRNAs.

In [ ]:
from pathlib import Path
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeElapsedColumn
from config import KOVERI_CONFIG, DEFAULT_OUTPUT_DIR
from utils.designer import design_koveri_primers

OUTPUT_DIR = DEFAULT_OUTPUT_DIR
INPUT_FASTA = OUTPUT_DIR / 'RNF12_gRNAs.fa'

config = dict(KOVERI_CONFIG)
config['min_informative_snps'] = 3
config['flank_steps'] = [250, 500, 750, 1000, 1500, 2500, 5000]
config['max_amplicon_size'] = 5000
config['blast_specificity'] = True
config['top_n_output'] = 20
config['specificity_candidates'] = 60

INPUT_FASTA, OUTPUT_DIR

Tune the values above when a region is difficult. The most common knobs are `min_informative_snps`, `flank_steps`, `max_amplicon_size`, and `blast_specificity`.

Top candidates are selected from the strictest passing post-Primer3 filter tier only. If one strict pair passes, the output is not padded with relaxed pairs.

In [ ]:
with Progress(
    SpinnerColumn(),
    TextColumn('[progress.description]{task.description}'),
    BarColumn(),
    TimeElapsedColumn(),
    transient=False,
) as progress:
    task = progress.add_task('Running KOveri primer design...', total=None)
    result = design_koveri_primers(INPUT_FASTA, OUTPUT_DIR, config)
    progress.update(task, description='KOveri primer design complete')
target = result['target']
print(f'Target interval: {target.chrom}:{target.start}-{target.end}')
print(f'Candidate pairs: {len(result["candidate_rows"])}')
print(f'Output directory: {result["output_dir"]}')
print(f'HTML report: {result["output_dir"] / "KOveri_report.html"}')

In [ ]:
import pandas as pd

guide_hits = pd.read_csv(OUTPUT_DIR / 'KOveri_guide_hits.csv')
guide_hits

In [ ]:
top = pd.read_csv(OUTPUT_DIR / 'KOveri_primer_top_candidates.csv')
display_cols = [
    'Chromosome', 'Amplicon_Start', 'Amplicon_End', 'Amplicon_Size',
    'SNP_Count_In_Amplicon', 'Left_Primer_Seq', 'Right_Primer_Seq',
    'Primer_Overlaps_SNP', 'Primer_Pair_Specificity_Pass'
]
top[display_cols]

In [ ]:
from IPython.display import IFrame, display

display(IFrame(src=str(OUTPUT_DIR / 'KOveri_report.html'), width='100%', height=700))

In [ ]:
print((OUTPUT_DIR / 'KOveri_summary.txt').read_text())